# 植物叶片病害识别 — Colab 训练笔记本

**用法**：
1. 打开 https://colab.research.google.com/ → 上传本文件
2. 菜单「代码执行程序」→「更改运行时类型」→ 硬件加速器选 **T4 GPU**
3. 从上往下依次运行每个代码块

**预计耗时**（T4 GPU，224px，batch 64）：
- ResNet18 15 epoch：约 25 分钟
- MobileNetV3-Large 15 epoch：约 22 分钟
- Swin-T 15 epoch：约 55 分钟

**注意**：Colab 免费版单次会话最长约 12 小时，闲置 90 分钟会断开。训练产物存在 Google Drive 才不会丢。

## 1. 检查 GPU

In [ ]:
import torch, sys
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('⚠️ 没有 GPU！请检查「运行时类型」是否选了 T4 GPU')

## 2. 挂载 Google Drive

数据集和训练结果都放 Drive 里，这样断线重连后不用重新下载。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/plant-leaf-disease'
os.makedirs(BASE, exist_ok=True)
print('工作目录:', BASE)

## 3. 下载数据集

从 HuggingFace 下载 PlantVillage（三个 parquet，合计约 813 MB）。

**如果 Drive 里已经有转换好的数据，跳过这一步**，直接跑第 4 块。

In [ ]:
!pip install -q datasets pillow tqdm

from datasets import load_dataset
import os

RAW = f'{BASE}/raw'
os.makedirs(RAW, exist_ok=True)

# 只下载，不转换（转换在下一步用我们的脚本做）
ds = load_dataset('GVJahnavi/PlantVillage_dataset', cache_dir=RAW)
print(ds)
print('类别数:', ds['train'].features['label'].num_classes)
print('训练集:', len(ds['train']), ' 测试集:', len(ds['test']))

## 4. 转换成 ImageFolder 格式

把 parquet 里的二进制图像写成 `train/val/test/<类别>/xxx.jpg` 结构。

**这一步较慢**（5.4 万张图写盘），约 10-15 分钟。转换完成后数据会存在 Drive 里，下次直接复用。

In [ ]:
import os
DATA = f'{BASE}/data'

if os.path.isdir(f'{DATA}/train') and len(os.listdir(f'{DATA}/train')) == 38:
    print('✅ 数据已存在，跳过转换')
else:
    from datasets import load_dataset
    from PIL import Image
    from pathlib import Path
    import io, random, json
    from collections import defaultdict

    ds = load_dataset('GVJahnavi/PlantVillage_dataset', cache_dir=f'{BASE}/raw')

    # 读类别名
    feat = ds['train'].features['label']
    if hasattr(feat, 'names'):
        names = feat.names
    elif isinstance(feat, dict) and 'names' in feat:
        names = feat['names']
    else:
        names = [f'class_{i}' for i in range(feat.num_classes)]

    def sanitize(s):
        return ''.join(c if (c.isalnum() or c in '_-') else '_' for c in s).strip('_')

    class_dirs = {i: f'{i:02d}_{sanitize(n)}' for i, n in enumerate(names)}
    random.seed(42)

    # 先把 train 按 9:1 分层切分
    by_class = defaultdict(list)
    for idx, row in enumerate(ds['train']):
        by_class[row['label']].append(idx)

    train_idx, val_idx = [], []
    for label, idxs in by_class.items():
        idxs = idxs[:]
        random.shuffle(idxs)
        n_val = max(1, int(len(idxs) * 0.1))
        val_idx.extend(idxs[:n_val])
        train_idx.extend(idxs[n_val:])

    print(f'train {len(train_idx)} / val {len(val_idx)} / test {len(ds["test"])}')

    def dump(split_name, indices, source):
        for label in class_dirs:
            os.makedirs(f'{DATA}/{split_name}/{class_dirs[label]}', exist_ok=True)
        n = 0
        for i in indices:
            row = source[i]
            label = row['label']
            img = row['image']
            if isinstance(img, dict):
                img = Image.open(io.BytesIO(img['bytes']))
            img = img.convert('RGB')
            out = f'{DATA}/{split_name}/{class_dirs[label]}/{i:06d}.jpg'
            img.save(out, quality=95)
            n += 1
            if n % 5000 == 0:
                print(f'  {split_name}: {n}/{len(indices)}')
        print(f'{split_name} 完成: {n} 张')

    dump('train', train_idx, ds['train'])
    dump('val', val_idx, ds['train'])
    dump('test', list(range(len(ds['test']))), ds['test'])

    with open(f'{DATA}/classes.json', 'w', encoding='utf-8') as f:
        json.dump([class_dirs[i] for i in range(len(names))], f, ensure_ascii=False, indent=2)
    print('✅ 转换完成')

## 5. 获取训练代码

从 GitHub 拉取仓库。**如果你改过代码，记得先 push 到 GitHub**。

In [ ]:
import os
REPO = '/content/plant-leaf-disease'
if os.path.isdir(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/wkylnj/plant-leaf-disease.git {REPO}

!ls {REPO}/ml/src

## 6. 冒烟测试（先跑这个，30 秒）

只用 5 个 batch 确认整条链路通。**不要跳过这一步直接开训**——如果数据路径、类别数、显存有问题，这里会立刻暴露，比训练半小时后崩溃好得多。

In [ ]:
%cd {REPO}/ml

!python src/train.py \
  --model resnet18 \
  --data-root {DATA} \
  --epochs 1 \
  --limit-batches 5 \
  --batch-size 32 \
  --num-workers 2 \
  --out-dir {BASE}/runs \
  --run-name smoke

## 7. 正式训练 — ResNet18（约 25 分钟）

这是主线模型。跑完后看 `summary.json` 里的 `test_acc`，预期 97%–99%。

In [ ]:
!python src/train.py \
  --model resnet18 \
  --data-root {DATA} \
  --epochs 15 \
  --batch-size 64 \
  --lr 3e-4 \
  --num-workers 4 \
  --out-dir {BASE}/runs \
  --run-name resnet18_e15

## 8. 正式训练 — MobileNetV3-Large（约 22 分钟）

轻量模型对照。参数量只有 ResNet18 的 38%，如果精度接近，就是报告里「轻量模型够用」的直接证据。

In [ ]:
!python src/train.py \
  --model mobilenet_v3_large \
  --data-root {DATA} \
  --epochs 15 \
  --batch-size 64 \
  --lr 3e-4 \
  --num-workers 4 \
  --out-dir {BASE}/runs \
  --run-name mobilenet_v3_large_e15

## 9. 正式训练 — Swin-T（约 55 分钟）

Transformer 结构对照。**如果时间紧张，这一项可以放到 Beta 阶段**，先保证前两个模型跑完。

In [ ]:
!python src/train.py \
  --model swin_t \
  --data-root {DATA} \
  --epochs 15 \
  --batch-size 32 \
  --lr 2e-4 \
  --num-workers 4 \
  --out-dir {BASE}/runs \
  --run-name swin_t_e15

## 10. 汇总所有结果

把每个 run 的 `summary.json` 读出来，生成一张对比表。这张表可以直接贴进报告。

In [ ]:
import json, glob, os

rows = []
for p in sorted(glob.glob(f'{BASE}/runs/*/summary.json')):
    with open(p, encoding='utf-8') as f:
        s = json.load(f)
    run = os.path.basename(os.path.dirname(p))
    rows.append((run, s))

print(f"{'run':32s} {'test_acc':>9s} {'macro_F1':>9s} {'params':>8s}")
print('-' * 62)
for run, s in rows:
    acc = s.get('test_acc') or s.get('accuracy') or 0
    f1 = s.get('test_macro_f1') or s.get('macro_f1') or 0
    prm = s.get('params_m') or 0
    print(f'{run:32s} {acc:9.4f} {f1:9.4f} {prm:7.1f}M')

# 导出成 markdown 表格
md = ['| 模型 | test acc | macro-F1 | 参数量 |', '|---|---:|---:|---:|']
for run, s in rows:
    acc = s.get('test_acc') or s.get('accuracy') or 0
    f1 = s.get('test_macro_f1') or s.get('macro_f1') or 0
    prm = s.get('params_m') or 0
    md.append(f'| {run} | {acc:.2%} | {f1:.2%} | {prm:.1f}M |')
table = '\n'.join(md)
print('')
print('=== 报告用 Markdown 表格 ===')
print(table)

with open(f'{BASE}/results_table.md', 'w', encoding='utf-8') as f:
    f.write(table)
print('')
print('已保存到', f'{BASE}/results_table.md')

## 11. 导出 ONNX（给后端用）

把最好的模型转成 ONNX，这样后端不需要装 PyTorch，用 onnxruntime 就能推理。

In [ ]:
!pip install -q onnx onnxruntime

import torch, onnx, json
import numpy as np
from models import build_model

RUN = 'resnet18_e15'   # 改这里换成别的模型
ckpt = f'{BASE}/runs/{RUN}/best.pt'
out_onnx = f'{BASE}/{RUN}.onnx'

model, meta = build_model(RUN.split('_e')[0], num_classes=38, pretrained=False)
state = torch.load(ckpt, map_location='cpu')
model.load_state_dict(state['model'] if 'model' in state else state)
model.eval()

dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy, out_onnx,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print('已导出', out_onnx)

# 验证误差
import onnxruntime as ort
sess = ort.InferenceSession(out_onnx, providers=['CPUExecutionProvider'])
with torch.no_grad():
    torch_out = model(dummy).numpy()
onnx_out = sess.run(None, {'input': dummy.numpy()})[0]
max_err = np.abs(torch_out - onnx_out).max()
print(f'PyTorch vs ONNX 最大误差: {max_err:.2e}')
print('✅ 通过' if max_err < 1e-4 else '❌ 误差过大，需检查')

# 顺手把类别名存下来
import shutil
shutil.copy(f'{DATA}/classes.json', f'{BASE}/classes.json')
print('类别名已复制到', f'{BASE}/classes.json')

## 12. 下载结果到本地

训练产物都在 Drive 里了。这一步把它们打包下载，或者你直接在 Drive 网页版里下载。

**报告只需要这三个东西**：
- `runs/*/summary.json`（精度数字）
- `runs/*/history.csv`（训练曲线）
- `results_table.md`（对比表）

In [ ]:
import shutil, glob, os

pack = '/content/results'
os.makedirs(pack, exist_ok=True)

for p in glob.glob(f'{BASE}/runs/*/summary.json'):
    run = os.path.basename(os.path.dirname(p))
    shutil.copy(p, f'{pack}/{run}_summary.json')
for p in glob.glob(f'{BASE}/runs/*/history.csv'):
    run = os.path.basename(os.path.dirname(p))
    shutil.copy(p, f'{pack}/{run}_history.csv')
if os.path.exists(f'{BASE}/results_table.md'):
    shutil.copy(f'{BASE}/results_table.md', f'{pack}/results_table.md')

shutil.make_archive('/content/results', 'zip', pack)
print('打包完成:', '/content/results.zip')

from google.colab import files
files.download('/content/results.zip')

---

## 常见问题

**Q: 报错 `CUDA out of memory`**
把 `--batch-size 64` 改成 `32` 或 `16`。Swin-T 显存占用比 ResNet18 大，已经默认设成 32。

**Q: 断线了怎么办**
重新挂载 Drive，数据还在。训练用 `--resume {BASE}/runs/xxx/last.pt` 续训。

**Q: 训练很慢，一个 epoch 超过 10 分钟**
检查是不是没用 GPU（第 1 块会打印）。如果是 CPU，去「运行时」→「更改运行时类型」重新选 T4。

**Q: 精度只有 60% 左右**
大概率是数据路径不对或类别数不匹配。跑第 6 块的冒烟测试，看输出的 `num_classes` 是不是 38。

**Q: 免费版额度不够**
优先保证 ResNet18 + MobileNetV3 两个跑完，Swin-T 可以放到 Beta 阶段。三个代表模型足以支撑「轻量 vs 重型」的对比结论。